# Exploring Survival on the Titanic
[https://www.kaggle.com/code/mrisdal/exploring-survival-on-the-titanic](https://www.kaggle.com/code/mrisdal/exploring-survival-on-the-titanic)

In [1]:
!pip install category_encoders

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

train = pd.read_csv('train.csv', na_values=["?", "NA", "missing", "nan"])
test = pd.read_csv('test.csv', na_values=["?", "NA", "missing", "nan"])
gender = pd.read_csv('gender_submission.csv')

df = pd.concat([train, test], sort=False)

train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [4]:
gender.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [5]:
df["Survived"] = df["Survived"].astype(bool)
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,False,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,True,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,True,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,True,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,False,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
413,1305,True,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,True,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,True,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,True,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [6]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df = df.drop(["SibSp", "Parch"], axis=1)

In [7]:
import re

def get_title(name):
    match = re.search(r',\s*([^\.]+)\.', name)
    if match:
        return match.group(1).strip()
    return None

df["Title"] = df["Name"].apply(get_title)
df["Title"] = df["Title"].replace(
    ["Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major",
     "Rev", "Sir", "Jonkheer", "Dona"], "Rare")
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})


import re

def get_ticket_prefix(ticket):
    match = re.match(r"([A-Za-z./]+)", ticket)
    if match:
        return match.group(1)
    return "None"

df["TicketPrefix"] = df["Ticket"].apply(get_ticket_prefix)

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,Ticket,Fare,Cabin,Embarked,FamilySize,IsAlone,Title,TicketPrefix
0,1,False,3,"Braund, Mr. Owen Harris",male,22.0,A/5 21171,7.2500,NaN,S,2,0,Mr,A/
1,2,True,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,PC 17599,71.2833,C85,C,2,0,Mrs,PC
2,3,True,3,"Heikkinen, Miss. Laina",female,26.0,STON/O2. 3101282,7.9250,NaN,S,1,1,Miss,STON/O
3,4,True,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,113803,53.1000,C123,S,2,0,Mrs,None
4,5,False,3,"Allen, Mr. William Henry",male,35.0,373450,8.0500,NaN,S,1,1,Mr,None


In [8]:
df["Cabin"].unique()

array([nan, 'C85', 'C123', 'E46', 'G6', 'C103', 'D56', 'A6',
       'C23 C25 C27', 'B78', 'D33', 'B30', 'C52', 'B28', 'C83', 'F33',
       'F G73', 'E31', 'A5', 'D10 D12', 'D26', 'C110', 'B58 B60', 'E101',
       'F E69', 'D47', 'B86', 'F2', 'C2', 'E33', 'B19', 'A7', 'C49', 'F4',
       'A32', 'B4', 'B80', 'A31', 'D36', 'D15', 'C93', 'C78', 'D35',
       'C87', 'B77', 'E67', 'B94', 'C125', 'C99', 'C118', 'D7', 'A19',
       'B49', 'D', 'C22 C26', 'C106', 'C65', 'E36', 'C54',
       'B57 B59 B63 B66', 'C7', 'E34', 'C32', 'B18', 'C124', 'C91', 'E40',
       'T', 'C128', 'D37', 'B35', 'E50', 'C82', 'B96 B98', 'E10', 'E44',
       'A34', 'C104', 'C111', 'C92', 'E38', 'D21', 'E12', 'E63', 'A14',
       'B37', 'C30', 'D20', 'B79', 'E25', 'D46', 'B73', 'C95', 'B38',
       'B39', 'B22', 'C86', 'C70', 'A16', 'C101', 'C68', 'A10', 'E68',
       'B41', 'A20', 'D19', 'D50', 'D9', 'A23', 'B50', 'A26', 'D48',
       'E58', 'C126', 'B71', 'B51 B53 B55', 'D49', 'B5', 'B20', 'F G63',
       'C62 C64',

In [9]:
df["Deck"] = df["Cabin"].str[0]
df["Deck"] = df["Deck"].fillna("Unknown")

print(df["Deck"].value_counts())

Deck
Unknown    1014
C            94
B            65
D            46
E            41
A            22
F            21
G             5
T             1
Name: count, dtype: int64


In [10]:
df = df.drop(["Name", "Ticket", "Cabin", "PassengerId"], axis=1)
df

,Survived,Pclass,Sex,Age,Fare,Embarked,FamilySize,IsAlone,Title,TicketPrefix,Deck
0,False,3,male,22.0,7.2500,S,2,0,Mr,A/,Unknown
1,True,1,female,38.0,71.2833,C,2,0,Mrs,PC,C
2,True,3,female,26.0,7.9250,S,1,1,Miss,STON/O,Unknown
3,True,1,female,35.0,53.1000,S,2,0,Mrs,None,C
4,False,3,male,35.0,8.0500,S,1,1,Mr,None,Unknown
...,...,...,...,...,...,...,...,...,...,...,...
413,True,3,male,NaN,8.0500,S,1,1,Mr,A.,Unknown
414,True,1,female,39.0,108.9000,C,1,1,Rare,PC,C
415,True,3,male,38.5,7.2500,S,1,1,Mr,SOTON/O.Q.,Unknown
416,True,3,male,NaN,8.0500,S,1,1,Mr,None,Unknown


In [11]:
import category_encoders as ce

encoder = ce.BinaryEncoder(cols=["TicketPrefix", "Deck"])

df = encoder.fit_transform(df)

df["Embarked"].fillna(df["Embarked"].mode()[0], inplace=True)
df = pd.get_dummies(df, columns=["Sex", "Embarked", "Title"], drop_first=True)

df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

df

C:\Users\User\AppData\Local\Temp\ipykernel_35292\4144683178.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Embarked"].fillna(df["Embarked"].mode()[0], inplace=True)


,Survived,Pclass,Age,Fare,FamilySize,IsAlone,TicketPrefix_0,TicketPrefix_1,TicketPrefix_2,TicketPrefix_3,...,Deck_2,Deck_3,Sex_male,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Title_the Countess
0,False,3,22.0,7.2500,2,0,0,0,0,0,...,0,1,True,False,True,False,True,False,False,False
1,True,1,38.0,71.2833,2,0,0,0,0,0,...,1,0,False,False,False,False,False,True,False,False
2,True,3,26.0,7.9250,1,1,0,0,0,0,...,0,1,False,False,True,True,False,False,False,False
3,True,1,35.0,53.1000,2,0,0,0,0,1,...,1,0,False,False,True,False,False,True,False,False
4,False,3,35.0,8.0500,1,1,0,0,0,1,...,0,1,True,False,True,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,True,3,28.0,8.0500,1,1,0,1,0,1,...,0,1,True,False,True,False,True,False,False,False
414,True,1,39.0,108.9000,1,1,0,0,0,0,...,1,0,False,False,False,False,False,False,True,False
415,True,3,38.5,7.2500,1,1,0,1,0,0,...,0,1,True,False,True,False,True,False,False,False
416,True,3,28.0,8.0500,1,1,0,0,0,1,...,0,1,True,False,True,False,True,False,False,False


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# keep only rows with labels
df_train = df[df["Survived"].notna()]

X = df_train.drop("Survived", axis=1)
y = df_train["Survived"]

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# apply PCA
pca = PCA(n_components=X.shape[0])
X_pca = pca.fit_transform(X)

print(pca.explained_variance_ratio_)


[9.41794165e-01 5.65662316e-02 8.37087927e-04 1.94615675e-04
 1.55764423e-04 9.33196137e-05 7.71568044e-05 6.08224998e-05
 3.64487485e-05 3.15935016e-05 2.73372164e-05 2.29524982e-05
 1.77341518e-05 1.64211465e-05 1.46192387e-05 1.40044180e-05
 1.27019944e-05 1.17042623e-05 7.55171280e-06 3.93989104e-06
 3.24552254e-06 4.46438290e-07 1.35308032e-07]


In [13]:
X_top = X_pca[:, :2]

In [14]:
# define a dictionary of classifiers to evaluate using cross-validation

from sklearn.model_selection import cross_val_score
from sklearn import svm
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

classifiers = {}
classifiers["DecisionTreeClassifier"] = tree.DecisionTreeClassifier()
classifiers["RandomForestClassifier"] = RandomForestClassifier()

for solver in ['lbfgs', 'sgd', 'adam']:
  for max_iter in [100, 500, 1000, 2000, 5000, 10000]:
    classifiers[f"MLPClassifier_{solver}_{max_iter}"] = MLPClassifier(solver=solver, max_iter=max_iter)

for c in [0.1, 1, 3, 10]:
    for kernel in ['linear','poly','rbf','sigmoid']:
        classifiers[f"svm_{c}_{kernel}"] = svm.SVC(kernel=kernel, C=c)

for k in range(2,30):
  for neighbors in range(2,10):
    classifiers[f"KNeighborsClassifier{k}_{neighbors}"] = KNeighborsClassifier(n_neighbors=neighbors, algorithm='kd_tree')

# classifiers["MultinomialNB"] = MultinomialNB()

for max_iter in [100, 500, 1000, 2000, 5000, 10000]:
  classifiers[f"LogisticRegression_{max_iter}"] = LogisticRegression(max_iter=max_iter)

In [ ]:
# evaluate each classifier using cross-validation and print the mean and standard deviation of the scores
max_score = 0
max_kernel = None

for name, clf in classifiers.items():
    clf_fit = clf.fit(X_train, y_train)
    cv_scores_temp = cross_val_score(clf_fit, X_top, y, cv=5)
    print(f"{name}: {cv_scores_temp.mean():.3f} (+/- {cv_scores_temp.std() * 2:.3f})")
    if cv_scores_temp.mean() > max_score:
        max_score = cv_scores_temp.mean()
        max_kernel = name

print(f"Best classifier: {max_kernel} with score {max_score:.3f}")

DecisionTreeClassifier: 0.586 (+/- 0.051)
RandomForestClassifier: 0.594 (+/- 0.055)


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to sc

MLPClassifier_lbfgs_100: 0.601 (+/- 0.045)


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to sc

MLPClassifier_lbfgs_500: 0.597 (+/- 0.040)


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want t

MLPClassifier_lbfgs_1000: 0.566 (+/- 0.061)


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want t